# IAU Project

## Faza 2 - Predspracovanie dát

In [40]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder

In [41]:

data_path = Path("Datasets/037")

files = [
    file
    for file in data_path.rglob("*")
    if file.is_file()
]


for file in files:
    print(file)

gateways = pd.read_csv(data_path / "gateways.csv",sep=';')
readings = pd.read_csv(data_path / "readings.csv",sep='\t')
units = pd.read_csv(data_path / "units.csv",sep=',')

Datasets/037/units.csv
Datasets/037/gateways.csv
Datasets/037/readings.csv


### Vstupné súbory používajú rozdielne oddeľovacie znaky. Z tohto dôvodu bol
### oddeľovač pre každý dataset zadaný explicitne

In [42]:
datasets = {
    "Gateways": gateways,
    "Readings": readings,
    "Units": units
}

for name, dataset in datasets.items():
    print(f"{name}: {dataset.shape}")
    print(dataset.columns.tolist())
    print()



display(gateways.head())
display(readings.head())
display(units.head())    

Gateways: (16000, 14)
['vendor', 'gatewayId', 'lat', 'seen_last', 'ipv4', 'versionFirmware', 'Technician', 'id_unit', 'modelHardware', 'id_depot', 'macAddress', 'Lon', 'dbm_signal', 'date_install']

Readings: (12618, 14)
['id_unit', 'ts_cycle', 'motorCurrent', 'filter_dp', 'oilTemperature', 'pressure_reservoir', 'pressure_working', 'pressureOil', 'vibrationRms', 'flowmeter', 'intake_humidity', 'ambientNoise', 'bearingTemp', 'failure']

Units: (2000, 7)
['Manufacturer', 'class_duty', 'id_unit', 'lastOverhaul', 'lineCode', 'id_depot', 'hoursService']



,vendor,gatewayId,lat,seen_last,ipv4,versionFirmware,Technician,id_unit,modelHardware,id_depot,macAddress,Lon,dbm_signal,date_install
0,"Holland, Moore and Foster",GW-00000,39.85515,25 Jan 2025,201.144.200.206,v3.4.1,Viviane Jopich,70.0,GWX-300,5,4b:57:9d:d8:85:7e,-8.87854,-67.0,2023-05-10
1,Castro Nascimento Lda.,GW-00001,39.97771,24 May 2024,162.167.208.12,v2.7.0,Klaus Peter Klemt-Klapp,463.0,GWX-200,67,28:1b:c1:bf:d2:72,-9.09701,-78.4,"04/18/2021, 00:00:00"
2,Green-Mitchell,GW-00002,39.68046,2024/10/21,214.16.189.57,v1.2.5,Steven Luna,759.0,NODE-9,109,2c:75:86:3d:87:96,-8.56949,-73.9,2021-10-22
3,Stey Fechner GbR,GW-00003,40.37614,09 Jul 2024,250.7.67.70,v3.4.1,Adriana Pinho,1966.0,GWX-100,31,50:33:0f:fe:aa:b6,-9.20827,-84.9,25 Apr 2021
4,Misicher Nette GmbH & Co. KGaA,GW-00004,40.07243,"10/24/2024, 00:00:00",225.244.98.178,v3.4.1,Mark Hines,457.0,NODE-7,34,01:f9:49:1a:95:c1,-6.35564,-71.6,2015/04/14


,id_unit,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,intake_humidity,ambientNoise,bearingTemp,failure
0,U01130,"02/10/2023, 17:12:00",4.36969,28.64924,49.67845,7.34282,7.31041,2.31187,4.99789,30.25891,66.67165,72.94257,64.95347,0
1,U00848,18 Sep 2023,4.22091,23.03530,41.77282,8.50849,8.53387,3.60240,1.87873,30.49048,89.24165,85.55312,74.46724,0
2,U00943,2023/12/29,2.80830,44.64527,69.78227,9.84619,9.70007,4.82881,4.49602,31.08300,80.81591,74.80889,60.05191,0
3,U01894,"11/12/2023, 15:24:00",4.45080,23.51547,47.49275,10.08831,10.00805,4.95817,2.15054,24.19140,17.74420,78.35942,67.82175,0
4,U00083,18 Nov 2023,3.97643,17.19827,70.25846,7.86674,7.88600,2.37735,4.11747,26.91899,19.60671,66.19757,59.54418,0


,Manufacturer,class_duty,id_unit,lastOverhaul,lineCode,id_depot,hoursService
0,Jäkel,L,1,05 Dec 2019,L6,12,13856.6
1,"Clark, Jackson and Anderson",H,2,"03/19/2023, 00:00:00",L4,109,14557.4
2,Pinto,1,3,2020-09-28,L6,26,15652.8
3,Fleming Group,N,4,2023/11/30,L6,68,NaN
4,Pinto,heavy,5,26 Feb 2018,L3,102,6141.3


In [43]:
print("Gateways columns:")
print(gateways.columns.tolist())

print("\nReadings columns:")
print(readings.columns.tolist())

print("\nUnits columns:")
print(units.columns.tolist())

Gateways columns:
['vendor', 'gatewayId', 'lat', 'seen_last', 'ipv4', 'versionFirmware', 'Technician', 'id_unit', 'modelHardware', 'id_depot', 'macAddress', 'Lon', 'dbm_signal', 'date_install']

Readings columns:
['id_unit', 'ts_cycle', 'motorCurrent', 'filter_dp', 'oilTemperature', 'pressure_reservoir', 'pressure_working', 'pressureOil', 'vibrationRms', 'flowmeter', 'intake_humidity', 'ambientNoise', 'bearingTemp', 'failure']

Units columns:
['Manufacturer', 'class_duty', 'id_unit', 'lastOverhaul', 'lineCode', 'id_depot', 'hoursService']


### since variable atribut failure is in the reading then reading is the base dataset, one measurement cycle of one unit at a particular ts_cycle

In [44]:
print(
    "Gateways and Readings:",
    set(gateways.columns) & set(readings.columns)
)

print(
    "Gateways and Units:",
    set(gateways.columns) & set(units.columns)
)

print(
    "Readings and Units:",
    set(readings.columns) & set(units.columns)
)

Gateways and Readings: {'id_unit'}
Gateways and Units: {'id_unit', 'id_depot'}
Readings and Units: {'id_unit'}


In [45]:
print("reading rows:", len(readings))

print(
    "Unique id_unit + ts_cycle:",
    readings[["id_unit", "ts_cycle"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Duplicated id_unit + ts_cycle:",
    readings.duplicated(
        subset=["id_unit", "ts_cycle"]
    ).sum()
)

reading rows: 12618
Unique id_unit + ts_cycle: 12413
Duplicated id_unit + ts_cycle: 205


### After observation, it was found out that one unit of measurements in reading is represented with a a unique combination of id_unit and ts_cycle. Rows where combination is not unique (205 rows)

In [46]:
print(
    "Completely duplicated rows:",
    readings.duplicated().sum()
)

Completely duplicated rows: 186


### 186 rows are complete duplicatedswhich represent the same measurement and 205 - 189 = 19 rows differ in at least one value.

In [47]:
readings_without_duplicates = readings.drop_duplicates()

conflicting_cycles = readings_without_duplicates[
    readings_without_duplicates.duplicated(
        subset=["id_unit", "ts_cycle"],
        keep=False
    )
].sort_values(["id_unit", "ts_cycle"])

print(
    "Rows involved in conflicting cycles:",
    len(conflicting_cycles)
)

print(
    "Number of conflicting cycle groups:",
    conflicting_cycles.groupby(
        ["id_unit", "ts_cycle"]
    ).ngroups
)

display(conflicting_cycles)

Rows involved in conflicting cycles: 38
Number of conflicting cycle groups: 19


,id_unit,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,intake_humidity,ambientNoise,bearingTemp,failure
2005,U00032,2023/10/12,4.35240,17.15376,45.35273,9.00545,8.89095,4.00633,6.54306,26.95236,79.81643,76.65725,75.41390,0
7681,U00032,2023/10/12,7.12740,15.12888,58.18034,8.97236,8.53935,4.26794,2.56571,23.35472,38.55386,67.42724,77.03787,0
2318,U00071,2023-10-07,8.28045,23.67242,63.31915,8.01949,8.27628,2.92703,2.31301,30.03363,59.87044,75.33242,68.78547,0
2905,U00071,2023-10-07,8.08684,16.92073,48.17299,8.53107,8.68286,3.49151,6.50042,30.06184,26.13176,72.54233,54.84770,0
3039,U00130,2023/01/05,5.99018,13.95925,57.64779,8.10456,8.26077,3.16462,5.01357,27.22841,30.01438,73.24225,62.83620,0
5424,U00130,2023/01/05,5.56193,20.75549,56.08689,9.57682,9.74196,4.50544,4.15699,23.99710,56.90525,75.14271,64.50457,0
8643,U00131,2023-08-21,6.26486,27.40713,38.63368,9.58495,9.39346,4.56932,1.99188,19.52254,73.32488,76.94241,77.65421,0
9172,U00131,2023-08-21,4.78847,24.34563,67.91887,8.46736,8.62370,3.57375,1.79134,29.24994,89.25581,70.56869,50.19603,0
5317,U00138,2023/05/19,5.60416,17.83368,51.86838,9.56384,9.68258,4.67103,4.61489,27.58880,10.61184,74.46813,67.06151,0
5679,U00138,2023/05/19,4.34585,31.33845,60.86035,8.41328,8.29496,3.90827,3.54085,32.16959,35.58525,77.72031,68.02897,0


In [48]:
# Gateways

gateways['seen_last'] = pd.to_datetime(gateways['seen_last'], format="mixed", errors="coerce")
gateways['date_install'] = pd.to_datetime(gateways['date_install'], format="mixed", errors="coerce")
gateways.loc[gateways["seen_last"] < gateways["date_install"], 'seen_last'] = pd.NaT
gateways["id_unit"] = gateways["id_unit"].astype("Int64")

# Readings

readings["id_unit"] = readings["id_unit"].str.extract(r"(\d+)")[0].astype(int)
readings["ts_cycle"] = pd.to_datetime(readings["ts_cycle"], format="mixed")
readings.loc[readings['motorCurrent'] == 99, 'motorCurrent'] = np.nan
readings.loc[readings['oilTemperature'] == -999, 'oilTemperature'] = np.nan
readings = readings.drop_duplicates().reset_index(drop=True)
for c in ["filter_dp", "vibrationRms"]:
    readings[f"{c}_missing"] = readings[c].isna().astype(int)


# Units 
units['lastOverhaul'] = pd.to_datetime(units['lastOverhaul'], format="mixed", errors="coerce")
DUTY_MAP = {
    "l": 1, "light": 1, "1": 1,
    "n": 2, "normal": 2, "med": 2, "2": 2,
    "h": 3, "heavy": 3, "3": 3,
}

duty_clean = units["class_duty"].astype("string").str.strip().str.lower()
units["class_duty"] = duty_clean.map(DUTY_MAP).astype("Int64")

### most of the measurements have the same failure value, and only in one combination of id_unit + ts_cycle this value differs. Complete duplicates will be removed, but since ts_cycle represent only the data, non-identical rows will remain. (supposing the same measurement was took more than one time on the same day)

In [49]:
print("Number of rows in units:", len(units))
print("Number of unique id_unit values:", units["id_unit"].nunique())
print(
    "Number of duplicated id_unit values:",
    units["id_unit"].duplicated().sum()
)
print(
    "Number of missing id_unit values:",
    units["id_unit"].isna().sum()
)

Number of rows in units: 2000
Number of unique id_unit values: 2000
Number of duplicated id_unit values: 0
Number of missing id_unit values: 0


In [50]:
units.head()

,Manufacturer,class_duty,id_unit,lastOverhaul,lineCode,id_depot,hoursService
0,Jäkel,1,1,2019-12-05,L6,12,13856.6
1,"Clark, Jackson and Anderson",3,2,2023-03-19,L4,109,14557.4
2,Pinto,1,3,2020-09-28,L6,26,15652.8
3,Fleming Group,2,4,2023-11-30,L6,68,NaN
4,Pinto,3,5,2018-02-26,L3,102,6141.3


In [51]:
readings.head()

,id_unit,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,intake_humidity,ambientNoise,bearingTemp,failure,filter_dp_missing,vibrationRms_missing
0,1130,2023-02-10 17:12:00,4.36969,28.64924,49.67845,7.34282,7.31041,2.31187,4.99789,30.25891,66.67165,72.94257,64.95347,0,0,0
1,848,2023-09-18 00:00:00,4.22091,23.03530,41.77282,8.50849,8.53387,3.60240,1.87873,30.49048,89.24165,85.55312,74.46724,0,0,0
2,943,2023-12-29 00:00:00,2.80830,44.64527,69.78227,9.84619,9.70007,4.82881,4.49602,31.08300,80.81591,74.80889,60.05191,0,0,0
3,1894,2023-11-12 15:24:00,4.45080,23.51547,47.49275,10.08831,10.00805,4.95817,2.15054,24.19140,17.74420,78.35942,67.82175,0,0,0
4,83,2023-11-18 00:00:00,3.97643,17.19827,70.25846,7.86674,7.88600,2.37735,4.11747,26.91899,19.60671,66.19757,59.54418,0,0,0


### id-unit formats differe so they nee to be change dinto the same format

In [52]:
units["id_unit_key"] = pd.to_numeric(
    units["id_unit"],
    errors="coerce"
).astype("Int64")

In [53]:
readings["id_unit_key"] = pd.to_numeric(
    readings["id_unit"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.removeprefix("U"),
    errors="coerce"
).astype("Int64")

In [54]:
display(
    units[
        ["id_unit", "id_unit_key"]
    ].head(10)
)

display(
    readings[
        ["id_unit", "id_unit_key"]
    ].head(10)
)

,id_unit,id_unit_key
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5
5,6,6
6,7,7
7,8,8
8,9,9
9,10,10


,id_unit,id_unit_key
0,1130,1130
1,848,848
2,943,943
3,1894,1894
4,83,83
5,777,777
6,688,688
7,1370,1370
8,247,247
9,946,946


### in both datasets id_unit were assigned type int

In [55]:
print("Number of rows in units:", len(units))
print(
    "Unique normalized units:",
    units["id_unit_key"].nunique()
)
print(
    "Duplicated normalized units:",
    units["id_unit_key"].duplicated().sum()
)
print(
    "Missing normalized units:",
    units["id_unit_key"].isna().sum()
)

Number of rows in units: 2000
Unique normalized units: 2000
Duplicated normalized units: 0
Missing normalized units: 0


In [56]:
reading_unit_ids = set(readings["id_unit_key"])
unit_ids = set(units["id_unit_key"])

missing_in_units = reading_unit_ids - unit_ids

print(
    "Reading units not found in units:",
    len(missing_in_units)
)

print(sorted(missing_in_units)[:20])

Reading units not found in units: 111
[np.int64(90001), np.int64(90002), np.int64(90005), np.int64(90010), np.int64(90030), np.int64(90035), np.int64(90054), np.int64(90077), np.int64(90081), np.int64(90083), np.int64(90092), np.int64(90093), np.int64(90099), np.int64(90101), np.int64(90102), np.int64(90112), np.int64(90119), np.int64(90120), np.int64(90125), np.int64(90126)]


### 111 valus for id-unit in reading do not have corresponding row in units dataset

In [57]:
missing_unit_mask = readings["id_unit_key"].isin(missing_in_units)

print(
    "Unique missing units:",
    readings.loc[missing_unit_mask, "id_unit_key"].nunique()
)

print(
    "Reading rows with no matching unit:",
    missing_unit_mask.sum()
)



Unique missing units: 111
Reading rows with no matching unit: 121


### since id_unit in reading can be shared among other measurement, despite of 111 unique value which are not included in units, there are also duplicated of these rows which makes the number of rows without a data assigned from unit - 122

In [58]:
display(
    readings.loc[missing_unit_mask]
    .sort_values(["id_unit_key", "ts_cycle"])
    .head(30)
)

,id_unit,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,intake_humidity,ambientNoise,bearingTemp,failure,filter_dp_missing,vibrationRms_missing,id_unit_key
9463,90001,2023-01-08 00:00:00,4.61808,16.91618,61.64541,8.99371,8.94367,3.47438,8.47014,32.96959,51.79001,89.53124,57.81237,0,0,0,90001
3460,90001,2023-12-23 00:00:00,9.23093,29.23132,52.29549,9.63200,9.64805,4.26823,0.85261,33.13956,46.48159,86.04681,71.06732,0,0,0,90001
3373,90002,2023-02-12 00:00:00,2.97062,9.79832,44.89764,8.39591,8.54568,3.13107,1.94272,26.57043,47.27167,65.55677,81.45000,0,0,0,90002
6165,90002,2023-11-14 00:00:00,3.24440,20.26538,52.76692,8.74667,8.89841,3.67865,2.52919,29.12073,27.69169,70.42998,66.86160,0,0,0,90002
6448,90005,2023-09-19 00:00:00,7.54856,21.28621,44.10811,8.26120,8.24875,3.19977,2.04213,22.70890,85.49729,68.74093,57.81270,0,0,0,90005
10751,90010,2023-07-24 00:00:00,4.20566,27.00763,51.15061,8.34128,8.52876,3.46635,3.24375,23.98106,23.36035,74.78219,71.28933,0,0,0,90010
5081,90030,2023-12-14 01:30:00,5.55186,28.69114,48.52296,9.36780,9.30803,4.31523,3.61312,26.94594,67.86193,78.09257,71.34777,0,0,0,90030
3109,90035,2023-05-02 00:00:00,4.44077,12.38933,51.01765,9.01408,9.05445,4.36976,4.00167,30.29333,64.42437,75.47371,58.53679,0,0,0,90035
6229,90054,2023-12-22 00:00:00,6.59299,29.62719,60.40211,8.65615,8.73558,4.12839,3.97299,35.22304,63.00646,76.60263,68.11696,0,0,0,90054
11858,90077,2023-09-03 00:00:00,2.90312,21.11323,45.49549,8.56610,8.60802,3.22805,1.25545,30.52726,32.26283,71.76824,74.58435,0,0,0,90077


In [59]:
print(
    "1 exists:",
    (readings["id_unit_key"] == 1).any()
)

print(
    "90001 exists:",
    (readings["id_unit_key"] == 90001).any()
)

1 exists: True
90001 exists: True


### all values without a corresponding data from units start with 9 so we checked whether U90001 can be relate to 1. As a cocnlusion they are not relate isnce there are both values in a dataset

In [60]:
readings_clean = readings.drop_duplicates().copy()

data = readings_clean.merge(
    units,
    on="id_unit_key",
    how="left",
    validate="many_to_one",
    suffixes=("_reading", "_unit")
)

In [61]:
print("Readings before merge:", len(readings_clean))
print("Rows after merge:", len(data))

Readings before merge: 12432
Rows after merge: 12432


In [62]:
data.head(10)



,id_unit_reading,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,...,filter_dp_missing,vibrationRms_missing,id_unit_key,Manufacturer,class_duty,id_unit_unit,lastOverhaul,lineCode,id_depot,hoursService
0,1130,2023-02-10 17:12:00,4.36969,28.64924,49.67845,7.34282,7.31041,2.31187,4.99789,30.25891,...,0,0,1130,Barth Gute OHG mbH,2,1130.0,2020-02-17,L5,40.0,6665.4
1,848,2023-09-18 00:00:00,4.22091,23.03530,41.77282,8.50849,8.53387,3.60240,1.87873,30.49048,...,0,0,848,Higgins Inc,1,848.0,2022-06-03,L3,27.0,NaN
2,943,2023-12-29 00:00:00,2.80830,44.64527,69.78227,9.84619,9.70007,4.82881,4.49602,31.08300,...,0,0,943,Nette Weller Stiftung & Co. KGaA,1,943.0,2019-10-14,L6,14.0,16720.9
3,1894,2023-11-12 15:24:00,4.45080,23.51547,47.49275,10.08831,10.00805,4.95817,2.15054,24.19140,...,0,0,1894,Striebitz AG,2,1894.0,2019-02-02,L2,69.0,12480.2
4,83,2023-11-18 00:00:00,3.97643,17.19827,70.25846,7.86674,7.88600,2.37735,4.11747,26.91899,...,0,0,83,Hunt LLC,2,83.0,2017-10-09,L3,48.0,9389.6
5,777,2023-07-06 00:00:00,3.22686,28.24065,60.57740,9.00718,9.20808,4.48304,3.30593,32.48052,...,0,0,777,Castro Nascimento Lda.,1,777.0,2022-11-24,L3,39.0,8219.0
6,688,2023-09-09 00:00:00,2.76081,15.84201,60.89518,9.15108,9.37328,3.61046,2.11979,25.28018,...,0,0,688,Pinto,1,688.0,2019-08-10,L6,1.0,10525.3
7,1370,2023-09-20 00:00:00,4.50663,19.46103,42.76148,8.03207,8.31141,3.43468,1.54556,21.92854,...,0,0,1370,Moreira Henriques S.A.,1,1370.0,2023-09-29,L6,111.0,16085.6
8,247,2023-09-25 00:00:00,4.64585,41.59096,71.94085,8.59036,8.89304,3.66107,1.82557,30.47198,...,0,0,247,Börner Klemt KG,2,247.0,2016-12-21,L5,18.0,12930.7
9,946,2023-12-14 00:00:00,9.18168,37.39852,52.08046,8.02453,8.24464,2.78188,5.34202,33.58937,...,0,0,946,Salazar-Harris,3,946.0,2021-06-15,L3,119.0,9942.1


In [63]:
display(
    data[data["Manufacturer"].isna()]
)

,id_unit_reading,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,...,filter_dp_missing,vibrationRms_missing,id_unit_key,Manufacturer,class_duty,id_unit_unit,lastOverhaul,lineCode,id_depot,hoursService
113,90081,2023-11-09 00:00:00,8.09272,24.24770,60.49406,8.47530,8.54376,3.12272,1.17610,22.35824,...,0,0,90081,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
130,90396,2023-04-03 00:00:00,4.87806,38.56177,79.16946,8.79828,8.50186,3.68216,1.52900,33.10192,...,0,0,90396,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
264,90357,2023-03-07 05:01:00,9.03454,26.26709,53.02407,8.70299,8.66560,3.86911,1.26883,21.94601,...,0,0,90357,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
293,90323,2023-09-10 00:00:00,7.87523,26.32598,15.00000,8.87393,8.78140,3.68672,1.75551,29.95337,...,0,0,90323,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
385,90270,2023-01-11 00:00:00,4.67702,26.61143,36.07523,8.61352,8.59516,3.78436,1.02511,26.62764,...,0,0,90270,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12126,90391,2023-10-08 00:00:00,6.98345,14.84817,23.02443,8.73671,8.67595,3.31412,2.34774,28.75813,...,0,0,90391,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
12144,90235,2023-07-19 00:00:00,3.16738,23.03618,52.21291,8.39977,8.55105,4.10023,2.26784,35.33094,...,0,0,90235,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
12247,90210,2023-07-01 04:42:00,4.90359,10.49898,57.99938,9.44626,9.35361,3.28957,1.63341,30.71579,...,0,0,90210,NaN,<NA>,NaN,NaT,NaN,NaN,NaN
12255,90396,2023-04-04 00:00:00,4.47671,19.62528,77.34806,8.02995,8.09985,2.97059,3.37861,34.76185,...,0,0,90396,NaN,<NA>,NaN,NaT,NaN,NaN,NaN


### we performed left merge to avoid getting rid of measurements which do not have metadata from units

In [64]:
metadata_missing = data["Manufacturer"].isna()

print(
    "Rows without unit metadata:",
    metadata_missing.sum()
)

print(
    "Unique units without metadata:",
    data.loc[
        metadata_missing,
        "id_unit_key"
    ].nunique()
)

Rows without unit metadata: 121
Unique units without metadata: 111


In [65]:
print("id_unit type:", gateways["id_unit"].dtype)
print("gatewayId type:", gateways["gatewayId"].dtype)

print("\nid_unit examples:")
print(gateways["id_unit"].head(10).tolist())

print("\ngatewayId examples:")
print(gateways["gatewayId"].head(10).tolist())

id_unit type: Int64
gatewayId type: str

id_unit examples:
[70, 463, 759, 1966, 457, 1898, 333, 360, 320, 813]

gatewayId examples:
['GW-00000', 'GW-00001', 'GW-00002', 'GW-00003', 'GW-00004', 'GW-00005', 'GW-00006', 'GW-00007', 'GW-00008', 'GW-00009']


### possible shared value is id_unit so we cahnge it into int

In [66]:
gateways["id_unit_key"] = pd.to_numeric(
    gateways["id_unit"],
    errors="coerce"
).astype("Int64")


display(
    gateways[
        ["id_unit", "id_unit_key"]
    ].head(10)
)

print(
    "Missing original id_unit:",
    gateways["id_unit"].isna().sum()
)

print(
    "Missing normalized id_unit:",
    gateways["id_unit_key"].isna().sum()
)

,id_unit,id_unit_key
0,70,70
1,463,463
2,759,759
3,1966,1966
4,457,457
5,1898,1898
6,333,333
7,360,360
8,320,320
9,813,813


Missing original id_unit: 14000
Missing normalized id_unit: 14000


In [67]:
gateways_with_unit = gateways.dropna(
    subset=["id_unit_key"]
).copy()

print(
    "Gateway rows with unit:",
    len(gateways_with_unit)
)

print(
    "Unique units:",
    gateways_with_unit["id_unit_key"].nunique()
)

print(
    "Duplicated units:",
    gateways_with_unit["id_unit_key"].duplicated().sum()
)

print(
    "Unique gateway IDs:",
    gateways_with_unit["gatewayId"].nunique()
)

Gateway rows with unit: 2000
Unique units: 2000
Duplicated units: 0
Unique gateway IDs: 2000


In [68]:
data_unit_ids = set(data["id_unit_key"].dropna())
gateway_unit_ids = set(
    gateways_with_unit["id_unit_key"]
)

shared_gateway_ids = data_unit_ids & gateway_unit_ids
only_in_data = data_unit_ids - gateway_unit_ids
only_in_gateways = gateway_unit_ids - data_unit_ids

print("Shared unit IDs:", len(shared_gateway_ids))
print("Units in data without gateway:", len(only_in_data))
print("Gateway units without readings:", len(only_in_gateways))

Shared unit IDs: 1998
Units in data without gateway: 111
Gateway units without readings: 2


In [69]:
print("Gateway unit IDs without readings:")
print(sorted(only_in_gateways))

Gateway unit IDs without readings:
[np.int64(1077), np.int64(1632)]


In [70]:
display(
    gateways_with_unit[
        gateways_with_unit["id_unit_key"].isin([1077, 1632])
    ]
)

,vendor,gatewayId,lat,seen_last,ipv4,versionFirmware,Technician,id_unit,modelHardware,id_depot,macAddress,Lon,dbm_signal,date_install,id_unit_key
1502,Morales Ltd,GW-01502,40.25524,2024-05-01,24.147.205.239,v3.4.1,Manuel Ferreira,1632,GWX-100,18,7a:6e:45:79:f6:a5,-7.13336,-67.8,2017-11-15,1632
1630,Maia,GW-01630,39.14850,2024-02-13,65.220.63.115,v3.4.1,Amanda Mcneil,1077,NODE-9,9,ef:52:b7:f4:ab:39,-8.44468,-72.5,2015-01-18,1077


In [71]:
display(
    readings[
        readings["id_unit_key"].isin([1077, 1632])
    ]
)

,id_unit,ts_cycle,motorCurrent,filter_dp,oilTemperature,pressure_reservoir,pressure_working,pressureOil,vibrationRms,flowmeter,intake_humidity,ambientNoise,bearingTemp,failure,filter_dp_missing,vibrationRms_missing,id_unit_key


### two measurements does not have an id which can be assigned to reading id so we dont consider them

In [72]:
gateway_columns = [
    "id_unit_key",
    "gatewayId",
    "vendor",
    "lat",
    "Lon",
    "seen_last",
    "ipv4",
    "versionFirmware",
    "Technician",
    "modelHardware",
    "id_depot",
    "macAddress",
    "dbm_signal",
    "date_install"
]

In [73]:
rows_before_gateway_merge = len(data)

data = data.merge(
    gateways_with_unit[gateway_columns],
    on="id_unit_key",
    how="left",
    validate="many_to_one",
    suffixes=("_unit", "_gateway")
)

print("Rows before merge:", rows_before_gateway_merge)
print("Rows after merge:", len(data))

Rows before merge: 12432
Rows after merge: 12432


In [74]:
print(
    "Rows without gateway metadata:",
    data["gatewayId"].isna().sum()
)

print(
    "Unique units without gateway metadata:",
    data.loc[
        data["gatewayId"].isna(),
        "id_unit_key"
    ].nunique()
)

Rows without gateway metadata: 121
Unique units without gateway metadata: 111


# Split data (Train/Validate)

In [75]:
X = data.drop(columns = 'failure')
y = data['failure']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify=y, random_state=42
)

# Pipe line for data trasnformation
## Type fixer class created to convert date collumns from string to pd_datetime but for machine learning dates are usless so we need to convert it to number

In [76]:

class TypeFixer(BaseEstimator, TransformerMixin):
    date_cols = ["ts_cycle", "lastOverhaul", "seen_last", "date_install"]                                                                                               
    drop_cols = ["id_unit", "id_unit_key", "gatewayId", "ipv4", "macAddress", "Technician"]
    sentinels = {"oilTemperature": [-999], "motorCurrent": [99]}

    def fit(self, X, y = None):
        return self

    def transform(self, X):
        X = X.copy()
        X["motorCurrent_rel"] = X["motorCurrent"] - X.groupby("class_duty")["motorCurrent"].transform("median")
        for col, value in self.sentinels.items():
            X[col] = X[col].replace(value, np.nan)
        for col in self.date_cols:
            X[col] = pd.to_datetime(X[col], format="mixed", errors="coerce")
        X["days_since_overhaul"] = (X["ts_cycle"] - X["lastOverhaul"]).dt.days  ### подумати як це зберігати                                                                               
        X["gateway_age_days"] = (X["ts_cycle"] - X["date_install"]).dt.days ### подумати як це зберігати                                                                                          
        X["days_since_seen"] = (X["seen_last"] - X["ts_cycle"]).dt.days  ### подумати як це зберігати                                                                                
        return X.drop(columns=self.date_cols + self.drop_cols, errors="ignore")


In [77]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    def __init__(self, low = 0.05, high = 0.95):
        self.low = low
        self.high = high

    def fit(self, X, y=None):
        self.low_ = np.nanquantile(X, self.low, axis=0)
        self.high_ = np.nanquantile(X, self.high, axis = 0)
        return self

    def transform(self, X):
        return np.clip(X, self.low_, self.high_)

In [79]:
num_cols = data.select_dtypes(include = 'number')
cat_cols = data.select_dtypes(include = 'str')

num_pipe = Pipeline([
    ("clip", QuantileClipper(0.05, 0.95)),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num_pipe", num_pipe, num_cols),
    ("cat_pipe", cat_pipe, cat_cols),
])

full_pipe = Pipeline([
    ("types", TypeFixer()),
    ("prep",  preprocess),
])